# **Method 1: Web Scraping (Rappler) & Method 2: YouTube API – Flood Control Project**
This notebook basically follows the same setup as the ones in class:

- **rappler_corpus.xlsx** → has a sheet called **rappler** with these columns: `title, link, date_published, text, source, like_count, reply_parent_id, cleaned_text`
- **youtube_corpus.xlsx** → has sheet called **youtube** with: `commenter, comment, commenter_profile_url, like_count, reply_parent_id, date_published, title, link, source, cleaned_text`
- **cleaned_corpus.xlsx** → has a sheet called **cleaned_corpus**, which combines both datasets using the columns they have in common

> Quick Notes
> - You'll need to actually run the internet stuff like requests/YouTube API on your own computer. This notebook just shows you how to do it step by step.
> - For YouTube links that have `&t=` in them, I  kept those in the `link` column as is. I pull out a separate `video_id` for when we need to use the API.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
FIG  = ROOT / "figures"

RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

print(f"RAW data will be saved to: {RAW}")

## Install & Imports

In [ ]:

# Install if needed
# !pip install requests beautifulsoup4 google-api-python-client pandas openpyxl langdetect emoji regex

import os, re, math, json, pandas as pd, numpy as np
from datetime import datetime
from typing import List, Dict, Any
import requests
from bs4 import BeautifulSoup
from langdetect import detect
from emoji import replace_emoji

OUTPUT_DIR = RAW # current folder
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Helper functions

In [ ]:

# Helpers
def basic_clean_text(s: str) -> str:
    if s is None: return ""
    s = str(s)
    s = re.sub(r"http\S+|www\.\S+", " ", s)           # for URLs
    s = replace_emoji(s, replace=" ")                    # for emojis
    s = re.sub(r"\s+", " ", s).strip()
    return s

def safe_lang(s: str) -> str:
    try:
        return detect(s)
    except Exception:
        return "unk"


## Provide URL lists here

In [ ]:

# URL lists
YOUTUBE_URLS = [
    "https://www.youtube.com/watch?v=F8SKKPwA4jI",
    "https://www.youtube.com/watch?v=aJkORsfAAW0",
    "https://www.youtube.com/watch?v=SssHh7UyzQg",
    "https://www.youtube.com/watch?v=0G1z6mXlbNU",
    "https://www.youtube.com/watch?v=qSlixyp_j44",
    "https://www.youtube.com/watch?v=eWLpqUp8u-k",
    "https://www.youtube.com/watch?v=q5rXGexLeCU",
    "https://www.youtube.com/watch?v=K4iKGhRo4_M",
    "https://www.youtube.com/watch?v=9zrpQ223kA8",
    "https://www.youtube.com/watch?v=qQD221b5aaE&t",
    "https://www.youtube.com/watch?v=Fc2slkvbVWU&t",
    "https://www.youtube.com/watch?v=965jgSl2n-M&t",
    "https://www.youtube.com/watch?v=9Zo1HAJL1r0",
    "https://www.youtube.com/watch?v=eMGV0-Sm4pA",
    "https://www.youtube.com/watch?v=7_6Ue-niMHQ",
    "https://www.youtube.com/watch?v=xQtR3p_7VaM",
    "https://www.youtube.com/watch?v=6xgvj6dYyJY",
    "https://www.youtube.com/watch?v=B_iOR2jmtEE",
    "https://www.youtube.com/watch?v=S05FYy2P0hg&t",
    "https://www.youtube.com/watch?v=tHWZQMqCvb0",
    "https://www.youtube.com/watch?v=GB1qiADEAHw",
    "https://www.youtube.com/watch?v=lOFNbMUOlxA",
    "https://www.youtube.com/watch?v=RtHnOxhw1DY",
    "https://www.youtube.com/watch?v=9or0sXYbfe8&t",
    "https://www.youtube.com/watch?v=Y7xz8PhrQU0",
    "https://www.youtube.com/watch?v=nnIhwntg5sY&t=2s",
    "https://www.youtube.com/watch?v=yyiSYt1NTVM&t=4s",
    "https://www.youtube.com/watch?v=965jgSl2n-M&t=4s",
    "https://www.youtube.com/watch?v=QA9MqXShkx4&t=3s",
    "https://www.youtube.com/watch?v=CTz8aDj5HNQ&t=3s",
    "https://www.youtube.com/watch?v=DET2AUmRmng&t=3s",
    "https://www.youtube.com/watch?v=r6TMQN_QbPw&t=2s",
    "https://www.youtube.com/watch?v=rSFcZOrhPrA&t=3s",
    "https://www.youtube.com/watch?v=qQD221b5aaE&t=6s",
    "https://www.youtube.com/watch?v=Fc2slkvbVWU&t=2s",
    "https://www.youtube.com/watch?v=jCcSFuu3CtE&t=3s",
]

RAPPLER_URLS = [
    "https://www.rappler.com/newsbreak/podcasts-videos/task-force-investigate-villar-properties-benefit-flood-control-projects/",
    "https://www.rappler.com/philippines/visayas/list-top-flood-control-contractors-towns-typhoon-tino-deadliest/",
    "https://www.rappler.com/philippines/metro-manila/flood-control-projects-approved-quezon-city-2021-2025/",
    "https://www.rappler.com/newsbreak/iq/list-status-cases-complaints-flood-control-corruption/",
    "https://www.rappler.com/philippines/map-flood-control-projects-metro-manila/",
    "https://www.rappler.com/philippines/n30235369-flood-control/",
    "https://www.rappler.com/philippines/vince-dizon-environmental-compliance-certificate-flood-control-project/",
    "https://www.rappler.com/philippines/ici-complaint-recommendation-bocaue-bulacan-flood-control-project/",
    "https://www.rappler.com/philippines/visayas/typhoon-tino-floods-expose-billions-failure-flood-control-sunstar-cebu/",
    "https://www.rappler.com/newsbreak/iq/what-is-flood-control-project/",
    "https://www.rappler.com/philippines/map-flood-control-projects-luzon/",
    "https://www.rappler.com/philippines/house-stops-infra-committee-probe-flood-control-projects/",
    "https://www.rappler.com/philippines/video-house-investigation-flood-control-projects-september-2-2025/",
    "https://www.rappler.com/voices/thought-leaders/opinion-flood-control-failure-environmental-destruction/",
    "https://www.rappler.com/philippines/luzon/coa-fraud-audit-report-ghost-substandard-flood-control-projects-bulacan-october-10-2025/",
    "https://www.rappler.com/voices/newsletters/be-the-good-tell-us-about-your-areas-flood-control-projects/",
    "https://www.rappler.com/philippines/dpwh-uncovers-ghost-flood-control-projects-october-9-2025/",
    "https://www.rappler.com/newsbreak/in-depth/christopher-co-rival-carlos-loria-flood-control-deals-bicol/",
    "https://www.rappler.com/newsbreak/iq/stories-analyses-explainers-flood-control-projects/",
    "https://www.rappler.com/newsbreak/iq/flood-control-budget-historical-data-dpwh/",
    "https://www.rappler.com/newsbreak/inside-track/flood-control-project-contractors-children-nepotism/",
    "https://www.rappler.com/voices/thought-leaders/vantage-point-purging-flood-control-rot-will-marcos-jr-sink-swim/",
    "https://www.rappler.com/philippines/oriental-mindoro-flood-control-controversy/",
    "https://www.rappler.com/newsbreak/podcasts-videos/flood-control-budget-cut-corruption/",
    "https://www.rappler.com/voices/thought-leaders/vantage-point-blockchain-drain-flood-control-corruption-swamp/",
    "https://www.rappler.com/philippines/thousands-march-protest-flood-control-corruption-september-21-2025/",
    "https://www.rappler.com/voices/thought-leaders/flood-control-latest-sign-government-greenwashing/",
    "https://www.rappler.com/philippines/map-flood-control-projects-mindanao/",
    "https://www.rappler.com/philippines/mindanao/pulangi-river-erosion-barmm-highway-flood-control-scandal/",
    "https://www.rappler.com/philippines/dpwh-scraps-local-funded-flood-control-projects-budget-proposal-2026/",
    "https://www.rappler.com/philippines/top-flood-control-contractors-mentioned-marcos-links-politicians-zaldy-co-discaya/",
    "https://www.rappler.com/philippines/dpwh-flood-control-projects-documents-destroyed-tampered/",
    "https://www.rappler.com/philippines/sentiment-corruption-government-infrastructure-octa-research-pulse-asia-survey-september-2025/",
    "https://www.rappler.com/newsbreak/podcasts-videos/dpwh-flood-control-billions-budget-allocation/",
    "https://www.rappler.com/philippines/bojie-dy-wants-house-committee-hearing-flood-control-suspended-september-22-2025/",
    "https://www.rappler.com/voices/thought-leaders/opinion-flood-control-tipping-point-for-filipinos/",
    "https://www.rappler.com/philippines/marcos-eyes-economic-sabotage-contractors-ghost-flood-control-projects-august-20-2025/",
]


## Scrape Rappler articles to `rappler_corpus.xlsx`

In [ ]:

# Rappler scraping
import time 
import numpy as np

HDRS = {"User-Agent": "Mozilla/5.0 (compatible; flood-research-bot; +http://example.com)"}

def fetch_rappler_article(url: str) -> dict:
    r = requests.get(url, headers=HDRS, timeout=25)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    
    if not soup.find("article"):
        print(f"Skipping (not article): {url}")
        return None
        
    title_el = soup.find("h1")
    title = title_el.get_text(strip=True) if title_el else ""
    
    time_el = soup.find("time")
    date_published = (time_el.get("datetime") or time_el.get_text(strip=True)) if time_el else ""
    
    text = " ".join(p.get_text(" ", strip=True) for p in soup.select("article p"))
    
    if not text.strip():
        print(f"Skipping (no body): {url}")
        return None

    row = {
        "title": title,
        "link": url,  # match  the'link', not 'url'
        "date_published": date_published, # match 'date_published', not 'date'
        "text": text,
        "source": "rappler",
        "like_count": np.nan,
        "reply_parent_id": np.nan,
        "cleaned_text": basic_clean_text(text)
    }
    return row

rappler_rows = []
for u in RAPPLER_URLS:
    try:
        row = fetch_rappler_article(u)
        if row:
            rappler_rows.append(row)
    except Exception as e:
        print(f"Failed to fetch {u}: {e}")
    
    time.sleep(0.8)

rappler_df = pd.DataFrame(rappler_rows)
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "rappler_corpus.xlsx"), engine="openpyxl") as xw:
    rappler_df.to_excel(xw, sheet_name="rappler", index=False)

print(f"Saved: rappler_corpus.xlsx rows: {len(rappler_df)}")

csv_path = os.path.join(OUTPUT_DIR, "rappler_corpus.csv")
rappler_df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Saved: {csv_path} rows: {len(rappler_df)}")

rappler_df.head(3)

## Collect YouTube comments to `youtube_corpus.xlsx`, while preserving `&t=` in link

In [ ]:

# YouTube (API)
def extract_video_id(url: str) -> str:
    m = re.search(r"(?:v=|be/)([\w-]{11})", url)
    return m.group(1) if m else ""

def get_youtube_client():
    from googleapiclient.discovery import build
    api_key = os.getenv("YOUTUBE_API_KEY")
    if not api_key:
        raise RuntimeError("Please set YOUTUBE_API_KEY environment variable.")
    return build("youtube","v3",developerKey=api_key)

def fetch_video_meta(youtube, video_ids: List[str]) -> dict:
    meta = {}
    for i in range(0, len(video_ids), 50):
        resp = youtube.videos().list(part="snippet,statistics", id=",".join(video_ids[i:i+50])).execute()
        for it in resp.get("items", []):
            vid = it["id"]
            sn = it["snippet"]
            meta[vid] = {
                "title": sn.get("title"),
                "channel": sn.get("channelTitle"),
            }
    return meta

def fetch_comments_for_video(youtube, video_id: str, max_comments=400) -> list:
    out = []
    pageToken = None
    fetched = 0
    while True and fetched < max_comments:
        resp = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=min(100, max_comments - fetched),
            pageToken=pageToken,
            textFormat="plainText",
        ).execute()
        for item in resp.get("items", []):
            sn = item["snippet"]["topLevelComment"]["snippet"]
            out.append({
                "commenter": sn.get("authorDisplayName"),
                "comment": sn.get("textDisplay"),
                "commenter_profile_url": sn.get("authorChannelUrl"),
                "like_count": sn.get("likeCount"),
                "reply_parent_id": np.nan,
                "date_published": sn.get("publishedAt"),
            })
            fetched += 1
        pageToken = resp.get("nextPageToken")
        if not pageToken:
            break
        time.sleep(0.2)
    return out

# De-duplication logic
# map of original URLs to video IDs, as before
video_map = {u: extract_video_id(u) for u in YOUTUBE_URLS}
# get the set of *unique* video IDs to fetch
unique_video_ids = sorted(list(set(v for v in video_map.values() if v)))

yt = get_youtube_client()
print(f"Fetching meta for {len(unique_video_ids)} unique video IDs...")
meta = fetch_video_meta(yt, unique_video_ids)

print(f"Fetching comments for {len(unique_video_ids)} unique video IDs...")
# fetch comments once per unique ID and store in a cache
comments_cache = {}
for vid in unique_video_ids:
    try:
        comments_cache[vid] = fetch_comments_for_video(yt, vid, max_comments=300)
    except Exception as e:
        print(f"Failed to fetch comments for {vid}: {e}")
        comments_cache[vid] = []
print("...comment fetching complete.")

# 4. build final rows by looping over the original map to preserve links
yt_rows = []
for orig_url, vid in video_map.items():
    if not vid: 
        continue
    
    # get metadata and comments from the cache
    title = meta.get(vid, {}).get("title", "")
    comments = comments_cache.get(vid, [])
    
    for c in comments:
        yt_rows.append({
            "commenter": c["commenter"],
            "comment": c["comment"],
            "commenter_profile_url": c["commenter_profile_url"],
            "like_count": c["like_count"],
            "reply_parent_id": c["reply_parent_id"],
            "date_published": c["date_published"],
            "title": title,
            "link": orig_url,
            "source": "youtube",
            "cleaned_text": basic_clean_text(c["comment"]),
            "video_id": vid
        })

youtube_df = pd.DataFrame(yt_rows)
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "youtube_corpus.xlsx"), engine="openpyxl") as xw:
    youtube_df.to_excel(xw, sheet_name="youtube", index=False)

csv_path = os.path.join(OUTPUT_DIR, "youtube_corpus.csv")
youtube_df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Saved: {csv_path} rows: {len(youtube_df)}")

youtube_df.head(3)

## Build `cleaned_corpus.xlsx` (union of  youtube and rappler)

In [ ]:
# Union: cleaned_corpus.xlsx (common schema)
common_cols = ["title","link","date_published","text","source","like_count","reply_parent_id","cleaned_text"]

rp_path = RAW / "rappler_corpus.xlsx"
yt_path = RAW / "youtube_corpus.xlsx"

rp = pd.read_excel(rp_path, sheet_name="rappler") if os.path.exists(rp_path) else pd.DataFrame()
yt = pd.read_excel(yt_path, sheet_name="youtube") if os.path.exists(yt_path) else pd.DataFrame()

# map YT rows into common schema (store comment as 'text')
yt_common = pd.DataFrame()
if not yt.empty:
    yt_common = yt.rename(columns={"comment":"text"})[["title","link","date_published","text","source","like_count","reply_parent_id","cleaned_text"]]

union = pd.concat([rp[common_cols] if not rp.empty else pd.DataFrame(columns=common_cols),
                   yt_common if not yt_common.empty else pd.DataFrame(columns=common_cols)],
                  ignore_index=True)

# this save path is correct
with pd.ExcelWriter(RAW / "cleaned_corpus.xlsx", engine="openpyxl") as xw:
    union.to_excel(xw, sheet_name="cleaned_corpus", index=False)

union.head(5)

# this save path is also correct
cleaned_csv_path = RAW / "cleaned_corpus.csv"
union.to_csv(cleaned_csv_path, index=False, encoding="utf-8")
print("Saved:", cleaned_csv_path, "rows:", len(union))